In [ ]:

import pandas as pd
import numpy as np 
import os
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

BASE_DIR = os.getcwd()
WINDOW_SIZE = 200


data_paths = [
        # ============================================================
        # ============================================================
        # tester2
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l02.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r02.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l02.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r02.csv"), # 5분
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l02.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r02.csv"), # 5분
]


def lowpass_array(x, fs=50.0, cutoff=5.0, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq

    b, a = butter(order, normal_cutoff, btype="low", analog=False)

    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()

    return filtfilt(b, a, x)

def load_and_preprocess_csv(
    file_path, skiprows=100, skipfooter=100, flag=False, zone=52, window_size=200
):

    df = pd.read_csv(
        file_path, skiprows=skiprows, skipfooter=skipfooter, engine="python"
    )

    df.columns = [
        "Time",
        "Accelerometer x",
        "Accelerometer y",
        "Accelerometer z",
        "Gyroscope x",
        "Gyroscope y",
        "Gyroscope z",
        "Magnetometer x",
        "Magnetometer y",
        "Magnetometer z",
        "Orientation x",
        "Orientation y",
        "Orientation z",
        "Pressure",
        "Latitude",
        "Longitude",
        "Altitude",
        "Speed_GPS",
    ]

    df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S.%f")
    start_dt = df["Time"].iloc[0]
    df["Elapsed Time"] = (df["Time"] - start_dt).dt.total_seconds()

    acc_cols = ["Accelerometer x", "Accelerometer y", "Accelerometer z"]
    gyro_cols = ["Gyroscope x", "Gyroscope y", "Gyroscope z"]

    # ================================
    # Acc RAW
    # ================================
    for c in acc_cols:
        df[c + "_raw"] = df[c]

    # ================================
    # Gyro RAW
    # ================================
    for c in gyro_cols:
        df[c + "_raw"] = df[c]

    # ================================
    # Acc LPF
    # ================================
    for c in acc_cols:
        df[c + "_lpf"] = lowpass_array(
            df[c].astype(float).to_numpy(),
            fs=50.0,
            cutoff=3.0,
            order=2,
        )

    # ================================
    # Gyro LPF
    # ================================
    for c in gyro_cols:
        df[c + "_lpf"] = lowpass_array(
            df[c].astype(float).to_numpy(),
            fs=50.0,
            cutoff=3.0,
            order=4,
        )

    # ================================
    # HP = RAW - LPF
    # ================================
    for c in acc_cols:
        df[c + "_hp"] = df[c + "_raw"] - df[c + "_lpf"]

    for c in gyro_cols:
        df[c + "_hp"] = df[c + "_raw"] - df[c + "_lpf"]

    # ================================
    # Norm (선택)
    # ================================
    df["Acc_Norm_raw"] = np.linalg.norm(
        df[[c + "_raw" for c in acc_cols]].values, axis=1
    )

    df["Acc_Norm_lpf"] = np.linalg.norm(
        df[[c + "_lpf" for c in acc_cols]].values, axis=1
    )

    df["Gyro_Norm_raw"] = np.linalg.norm(
        df[[c + "_raw" for c in gyro_cols]].values, axis=1
    )

    df["Gyro_Norm_lpf"] = np.linalg.norm(
        df[[c + "_lpf" for c in gyro_cols]].values, axis=1
    )

    return df


In [ ]:

def plot_gyro_lpf_segment(
    df,
    start=1000,
    end=2000,
    raw_cols=("Gyroscope x", "Gyroscope y", "Gyroscope z"),
    lpf_cols=("Gyroscope x_lpf", "Gyroscope y_lpf", "Gyroscope z_lpf"),
    title_prefix="Gyro LPF Comparison"
):
    """
    Gyro RAW vs LPF 구간 비교 플롯

    df : DataFrame
    start, end : 인덱스 구간
    raw_cols : 원본 gyro 컬럼들
    lpf_cols : LPF 적용된 gyro 컬럼들
    """

    for raw_c, lpf_c in zip(raw_cols, lpf_cols):
        plt.figure(figsize=(10, 4))
        plt.plot(df[raw_c].iloc[start:end], label="RAW", alpha=0.7)
        plt.plot(df[lpf_c].iloc[start:end], label="LPF", linewidth=2)
        plt.title(f"{title_prefix} – {raw_c} [{start}:{end}]")
        plt.xlabel("Index")
        plt.ylabel("rad/s")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()
        
def plot_acc_lpf_segment(
        df,
    start=1000,
    end=2000,
    raw_cols=("Accelerometer x", "Accelerometer y", "Accelerometer z"),
    lpf_cols=("Accelerometer x_lpf", "Accelerometer y_lpf", "Accelerometer z_lpf"),
    title_prefix="Acc LPF Comparison"
):
    for raw_c, lpf_c in zip(raw_cols, lpf_cols):
        plt.figure(figsize=(10, 4))
        plt.plot(df[raw_c].iloc[start:end], label="RAW", alpha=0.7)
        plt.plot(df[lpf_c].iloc[start:end], label="LPF", linewidth=2)
        plt.title(f"{title_prefix} – {raw_c} [{start}:{end}]")
        plt.xlabel("Index")
        plt.ylabel("m/s^2")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
df1 = load_and_preprocess_csv(data_paths[4])

plot_gyro_lpf_segment(df1, start=2000, end=3000)
plot_acc_lpf_segment(df1, start=1000, end=2000)